# Airflow Production — Executors, SLAs, Alerting, Monitoring

**Mental model**

Airflow is an orchestration control plane, not the data plane.  
Its production job is to schedule, track, retry, alert, and expose state for work happening across your platform.

**Local lab context**

- **Airflow**: `http://localhost:8082` using `apache/airflow:2.8.0`, `LocalExecutor`, credentials `admin/admin`
- **PostgreSQL**: `localhost:5432`, database `de_telemetry`, user `de_admin`
- **Kafka**: `localhost:9092`, container `citi_kafka`
- **Spark**: `pyspark==3.5.4`, `master=local[*]`
- **MLflow**: `http://localhost:5000`
- **dbt**: `C:/py_venv/proj_educate/Scripts/dbt.exe`
- **Databricks**: Serverless SQL Warehouse `b6657f31d1e7a179`
- **GCP**: project `citi-de-learning`
- **Azure**: subscription `b3811436-61fc-4a3a-a6a9-deb05955076d`
- **AWS**: profile `study`, region `us-east-1`, account `357811130281`

**Telemetry narrative**

Citi-style scenario: **6,000+ API endpoints** monitored for latency, error rate, and throughput, with alerts escalating through severity tiers.

**Source telemetry tables**

- `endpoints` — 10,000 rows
- `metrics` — 500,000 rows
- `alerts` — 25,000 rows

In [ ]:
from __future__ import annotations

import base64
import json
import os
import shutil
import subprocess
import sys
import textwrap
import time
from datetime import datetime, timedelta, timezone
from pathlib import Path
from typing import Any, Dict, List, Optional

import pandas as pd
import requests

AIRFLOW_BASE_URL = "http://localhost:8082"
AIRFLOW_USER = "admin"
AIRFLOW_PASSWORD = "admin"

POSTGRES = {
    "host": "localhost",
    "port": 5432,
    "database": "de_telemetry",
    "user": "de_admin",
    "password": "DeAdmin2026!",
}

STACK_CONTEXT = {
    "kafka": {"bootstrap_servers": "localhost:9092", "image": "confluentinc/cp-kafka:7.6.0", "container": "citi_kafka"},
    "spark": {"version": "3.5.4", "master": "local[*]", "JAVA_HOME": r"C:/Program Files/Java/jre1.8.0_481", "HADOOP_HOME": r"C:/hadoop"},
    "airflow": {"version": "2.8.0", "executor": "LocalExecutor", "url": AIRFLOW_BASE_URL},
    "mlflow": {"url": "http://localhost:5000", "backend": "SQLite"},
    "dbt": {"exe": r"C:/py_venv/proj_educate/Scripts/dbt.exe", "project": "citi_dbt", "target": "postgres"},
    "databricks": {"host": "https://dbc-9f35a83d-b4e7.cloud.databricks.com", "warehouse_id": "b6657f31d1e7a179"},
    "gcp": {"project": "citi-de-learning", "key": r"D:/Workspace/Technologies/_setup/gcp_key.json"},
    "azure": {"subscription": "b3811436-61fc-4a3a-a6a9-deb05955076d", "az_cli": r"C:\Program Files (x86)\Microsoft SDKs\Azure\CLI2\wbin\az.cmd"},
    "aws": {"profile": "study", "region": "us-east-1", "account": "357811130281"},
}

session = requests.Session()
session.auth = (AIRFLOW_USER, AIRFLOW_PASSWORD)
session.headers.update({"Content-Type": "application/json"})

def api_get(path: str, params: Optional[Dict[str, Any]] = None, timeout: int = 20) -> Dict[str, Any]:
    url = f"{AIRFLOW_BASE_URL}{path}"
    try:
        response = session.get(url, params=params, timeout=timeout)
        response.raise_for_status()
        if response.text:
            return response.json()
        return {"status": "ok", "url": url}
    except Exception as exc:
        return {
            "status": "error",
            "url": url,
            "error": str(exc),
        }

def api_post(path: str, payload: Optional[Dict[str, Any]] = None, timeout: int = 20) -> Dict[str, Any]:
    url = f"{AIRFLOW_BASE_URL}{path}"
    try:
        response = session.post(url, data=json.dumps(payload or {}), timeout=timeout)
        response.raise_for_status()
        if response.text:
            return response.json()
        return {"status": "ok", "url": url}
    except Exception as exc:
        return {
            "status": "error",
            "url": url,
            "payload": payload,
            "error": str(exc),
        }

def pretty(obj: Any) -> None:
    print(json.dumps(obj, indent=2, default=str))

health = api_get("/health")
version = api_get("/version")
print("Airflow health:")
pretty(health)
print("\nAirflow version:")
pretty(version)

## 3. Executor comparison

### LocalExecutor vs CeleryExecutor vs KubernetesExecutor

| Executor | Best fit | Strengths | Upgrade trigger |
|---|---|---|---|
| **LocalExecutor** | Single-node orchestration lab, small production footprint | Simple, low overhead, shared local scheduler/worker environment | Move beyond one machine, need stronger isolation or higher parallelism |
| **CeleryExecutor** | Medium-to-large distributed orchestration with persistent worker pool | Horizontal scale across workers, mature queue pattern | Need autoscaling, stronger per-task isolation, heterogeneous runtime profiles |
| **KubernetesExecutor** | Large-scale cloud-native orchestration | Each task as a pod, isolation, elasticity, easier multi-team tenancy | Default production pattern when scale, security boundaries, or ephemeral task execution matter |

### Citi-style decision rule

- **LocalExecutor** is perfect for this local learning stack.
- **CeleryExecutor** is the “classic distributed Airflow” pattern.
- **KubernetesExecutor** is the enterprise production pattern when every task should be isolated and independently scalable.

In [ ]:
config_payload = api_get("/api/v1/config")
executor_rows = []

if "sections" in config_payload:
    sections = config_payload.get("sections", [])
    for section in sections:
        if section.get("section") == "core":
            for option in section.get("options", []):
                if option.get("key") == "executor":
                    executor_rows.append(
                        {
                            "section": "core",
                            "key": "executor",
                            "value": option.get("value"),
                            "description": "Current Airflow executor from REST API config",
                        }
                    )

executor_df = pd.DataFrame(executor_rows or [{
    "section": "core",
    "key": "executor",
    "value": "unknown",
    "description": config_payload.get("error", "Could not read /api/v1/config from local Airflow"),
}])

display(executor_df)

upgrade_matrix = pd.DataFrame(
    [
        {"executor_type": "LocalExecutor", "use_case": "Single machine / local production-lite", "when_to_upgrade": "Need multiple workers or machine boundaries"},
        {"executor_type": "CeleryExecutor", "use_case": "Distributed worker pool", "when_to_upgrade": "Need pod-level isolation or burst autoscaling"},
        {"executor_type": "KubernetesExecutor", "use_case": "Cloud-native production", "when_to_upgrade": "Already top-tier for isolation + scale; tune cluster instead"},
    ]
)
display(upgrade_matrix)

## 4. SLA configuration

This section writes a small DAG with `sla=timedelta(minutes=1)` on a task, attempts to place it where Airflow can discover it, triggers the DAG, and polls the run state with a hard stop.

**Important production point**

- An SLA in Airflow is about **expected completion time**.
- `sla_miss_callback` lets you route misses to alerting systems.
- In real production, the callback usually goes to **email, Slack, PagerDuty, or an incident bus**.

In [ ]:
SLA_DAG_ID = "prod_sla_demo_v1"

dag_source = textwrap.dedent(f"""
from __future__ import annotations

from datetime import datetime, timedelta
import json

from airflow import DAG
from airflow.operators.python import PythonOperator


def _emit_context(**context):
    payload = {{
        "dag_id": context["dag"].dag_id,
        "task_id": context["task"].task_id,
        "run_id": context["run_id"],
        "logical_date": str(context["logical_date"]),
    }}
    print(json.dumps(payload, indent=2))


def _sleep_for_sla():
    import time
    time.sleep(70)
    print("Slept 70 seconds to demonstrate 1-minute SLA pressure.")


def _sla_miss_callback(dag, task_list, blocking_task_list, slas, blocking_tis):
    print("SLA MISS CALLBACK TRIGGERED")
    print("Tasks:", task_list)
    print("Blocking:", blocking_task_list)


with DAG(
    dag_id="{SLA_DAG_ID}",
    start_date=datetime(2026, 1, 1),
    schedule=None,
    catchup=False,
    default_args={{
        "owner": "sean.girgis",
        "retries": 1,
        "retry_delay": timedelta(minutes=1),
    }},
    tags=["production", "sla", "localexecutor"],
    sla_miss_callback=_sla_miss_callback,
) as dag:
    start = PythonOperator(
        task_id="start_context",
        python_callable=_emit_context,
    )

    slow_task = PythonOperator(
        task_id="slow_task",
        python_callable=_sleep_for_sla,
        sla=timedelta(minutes=1),
    )

    end = PythonOperator(
        task_id="finish_context",
        python_callable=_emit_context,
    )

    start >> slow_task >> end
""").strip() + "\n"

local_dag_dir_candidates = [
    Path.home() / "airflow" / "dags",
    Path.cwd() / "dags",
    Path("/opt/airflow/dags"),
]
written_locations = []

for dag_dir in local_dag_dir_candidates:
    try:
        dag_dir.mkdir(parents=True, exist_ok=True)
        dag_path = dag_dir / f"{SLA_DAG_ID}.py"
        dag_path.write_text(dag_source, encoding="utf-8")
        written_locations.append(str(dag_path))
    except Exception as exc:
        written_locations.append(f"{dag_dir} -> write_failed: {exc}")

docker_locations = []
docker_exe = shutil.which("docker")
if docker_exe:
    try:
        ps = subprocess.run([docker_exe, "ps", "--format", "{{.Names}}"], capture_output=True, text=True, check=True)
        containers = {line.strip() for line in ps.stdout.splitlines() if line.strip()}
        airflow_container = None
        for candidate in containers:
            if "airflow" in candidate.lower():
                airflow_container = candidate
                break

        if airflow_container:
            encoded = base64.b64encode(dag_source.encode("utf-8")).decode("ascii")
            cmd = (
                "python - <<'PY'\n"
                "import base64, pathlib\n"
                f"content = base64.b64decode('{encoded}').decode('utf-8')\n"
                f"path = pathlib.Path('/opt/airflow/dags/{SLA_DAG_ID}.py')\n"
                "path.write_text(content, encoding='utf-8')\n"
                "print(path)\n"
                "PY"
            )
            cp = subprocess.run([docker_exe, "exec", airflow_container, "bash", "-lc", cmd], capture_output=True, text=True)
            docker_locations.append({
                "container": airflow_container,
                "returncode": cp.returncode,
                "stdout": cp.stdout.strip(),
                "stderr": cp.stderr.strip(),
            })
    except Exception as exc:
        docker_locations.append({"docker_error": str(exc)})

print("DAG write attempts (local):")
for item in written_locations:
    print("-", item)

print("\nDAG write attempts (docker):")
for item in docker_locations or [{"info": "docker not available or no airflow container found"}]:
    print(item)

In [ ]:
refresh_info = api_get(f"/api/v1/dags/{SLA_DAG_ID}")
print("DAG lookup after write attempt:")
pretty(refresh_info)

unpause_info = api_post(f"/api/v1/dags/{SLA_DAG_ID}", {"is_paused": False})
print("\nUnpause attempt:")
pretty(unpause_info)

trigger_payload = {
    "dag_run_id": f"manual__{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')}",
    "note": "Triggered from production readiness notebook",
}
trigger_info = api_post(f"/api/v1/dags/{SLA_DAG_ID}/dagRuns", trigger_payload)
print("\nTrigger attempt:")
pretty(trigger_info)

dag_run_id = trigger_info.get("dag_run_id")
poll_history: List[Dict[str, Any]] = []
max_polls = 12
sleep_seconds = 10

if dag_run_id:
    for poll_num in range(1, max_polls + 1):
        state_info = api_get(f"/api/v1/dags/{SLA_DAG_ID}/dagRuns/{dag_run_id}")
        state = state_info.get("state", state_info.get("status", "unknown"))
        poll_history.append(
            {
                "poll_num": poll_num,
                "ts_utc": datetime.now(timezone.utc).isoformat(),
                "state": state,
            }
        )
        if state in {"success", "failed"}:
            break
        time.sleep(sleep_seconds)
else:
    poll_history.append(
        {
            "poll_num": 0,
            "ts_utc": datetime.now(timezone.utc).isoformat(),
            "state": "not_triggered",
            "note": trigger_info.get("error", "DAG trigger did not return dag_run_id"),
        }
    )

poll_df = pd.DataFrame(poll_history)
display(poll_df)

print("SLA concept:")
print("- slow_task uses sla=timedelta(minutes=1)")
print("- the task intentionally sleeps 70 seconds")
print("- in a live Airflow environment this can create an SLA miss record and invoke sla_miss_callback")

## 5. DAG health monitoring

The point is not just whether a DAG exists.  
The point is whether it is **healthy enough to trust**.

A simple red/yellow/green rule can be:

- **Green**: recent successes, no active failures
- **Yellow**: running now, or no recent runs, or mixed outcomes
- **Red**: recent failures dominate, or DAG is paused when it should be active

In [ ]:
dags_payload = api_get("/api/v1/dags", params={"limit": 100})
dags = dags_payload.get("dags", []) if isinstance(dags_payload, dict) else []

dag_records = []
for dag in dags[:25]:
    dag_id = dag.get("dag_id")
    runs_payload = api_get(f"/api/v1/dags/{dag_id}/dagRuns", params={"limit": 10, "order_by": "-execution_date"})
    runs = runs_payload.get("dag_runs", []) if isinstance(runs_payload, dict) else []
    state_counts = {}
    for run in runs:
        state = run.get("state", "unknown")
        state_counts[state] = state_counts.get(state, 0) + 1
    dag_records.append(
        {
            "dag_id": dag_id,
            "is_paused": dag.get("is_paused"),
            "total_recent_runs": len(runs),
            "running": state_counts.get("running", 0),
            "success": state_counts.get("success", 0),
            "failed": state_counts.get("failed", 0),
            "queued": state_counts.get("queued", 0),
        }
    )

dag_health_df = pd.DataFrame(dag_records)
display(dag_health_df.head(15))

def check_dag_health(row: pd.Series) -> str:
    if bool(row.get("failed", 0)) and row.get("failed", 0) >= max(1, row.get("success", 0)):
        return "red"
    if bool(row.get("is_paused", False)):
        return "yellow"
    if row.get("running", 0) > 0:
        return "yellow"
    if row.get("success", 0) > 0 and row.get("failed", 0) == 0:
        return "green"
    return "yellow"

if not dag_health_df.empty:
    dag_health_df["health"] = dag_health_df.apply(check_dag_health, axis=1)
    display(dag_health_df.sort_values(["health", "dag_id"]).head(20))
else:
    print("No DAG records returned from REST API.")

In [ ]:
dag_runs_payload = api_get("/api/v1/dagRuns", params={"limit": 100})
dag_runs = dag_runs_payload.get("dag_runs", []) if isinstance(dag_runs_payload, dict) else []

run_df = pd.DataFrame([
    {
        "dag_id": r.get("dag_id"),
        "dag_run_id": r.get("dag_run_id"),
        "state": r.get("state"),
        "start_date": r.get("start_date"),
        "end_date": r.get("end_date"),
    }
    for r in dag_runs
])

if not run_df.empty:
    summary = (
        run_df.groupby("state", dropna=False)
        .size()
        .reset_index(name="count")
        .sort_values("count", ascending=False)
    )
    display(summary)
    print("Total DAG runs observed:", len(run_df))
else:
    print("No DAG runs returned from /api/v1/dagRuns")

## 6. Connection management

**Rule:** credentials should live in Airflow connections, secrets backends, or environment-backed configuration — **not inside DAG code**.

- **Connection** = structured credential object (host, login, password, schema, port, extras)
- **Variable** = runtime parameter or small config value
- **Do not** store database passwords directly in operator source code

In [ ]:
connections_payload = api_get("/api/v1/connections", params={"limit": 100})
connections = connections_payload.get("connections", []) if isinstance(connections_payload, dict) else []

conn_df = pd.DataFrame([
    {
        "connection_id": c.get("connection_id"),
        "conn_type": c.get("conn_type"),
        "host": c.get("host"),
        "port": c.get("port"),
        "login": c.get("login"),
        "schema": c.get("schema"),
    }
    for c in connections
])

display(conn_df.head(20) if not conn_df.empty else pd.DataFrame([{"info": "No connections returned or endpoint unavailable"}]))

variable_vs_connection = pd.DataFrame(
    [
        {"object_type": "Airflow Variable", "best_for": "Feature flags, thresholds, non-secret config", "example": "alert_severity_threshold = high"},
        {"object_type": "Airflow Connection", "best_for": "Database/API credentials and structured endpoints", "example": "postgres_default for de_telemetry"},
    ]
)
display(variable_vs_connection)

example_connection_uri = (
    f"postgresql://{POSTGRES['user']}:***@{POSTGRES['host']}:{POSTGRES['port']}/{POSTGRES['database']}"
)
print("Example production-safe connection string rendering:")
print(example_connection_uri)
print("\nWhy not in DAG code?")
print("- credential rotation becomes painful")
print("- source control exposure risk rises")
print("- per-environment overrides become messy")

## 7. Production checklist

1. Choose the right **executor** for scale and isolation.
2. Tune **parallelism**, **dag_concurrency**, and worker capacity deliberately.
3. Set `catchup=False` unless historical backfill is explicitly intended.
4. Set `max_active_runs` to protect downstream systems.
5. Define a clear **retry policy**: retries, delay, exponential backoff if needed.
6. Use **SLA monitoring** and `sla_miss_callback` for critical workflows.
7. Wire **alerting** to email / Slack / PagerDuty, not just UI visibility.
8. Store logs in durable storage and make them searchable.
9. Clean up the **metadata DB** and old task logs on a schedule.
10. Version DAGs cleanly and deploy them with change control.

## 8. What just happened

- You queried Airflow’s REST API for config, DAGs, DAG runs, and connections.
- You compared **LocalExecutor**, **CeleryExecutor**, and **KubernetesExecutor** in operational terms.
- You generated a small SLA demo DAG with `sla=timedelta(minutes=1)` and a callback hook.
- You built a simple **red / yellow / green** health classifier for DAG status.
- You reviewed the production rule that **credentials belong in connections/secrets**, not in DAG code.

**Bottom line**

**LocalExecutor** works for single-machine orchestration.  
**KubernetesExecutor** is the Citi production pattern — each task is a pod, no shared state, scales to thousands of parallel tasks.